# 1. Define constants

In [6]:
import os
import re

# Define the relative path to the data directory
data_folder = "../data"
YEARS = []

# Iterate through all files in the data folder to find the years
for filename in os.listdir(data_folder):
    # Check if the file is a JPEG image
    if filename.endswith(".jpg"):
        # Use regular expression to find the first sequence of digits in the filename
        match = re.search(r"\d+", filename)
        
        # If a number is found, add it to the list
        if match:
            YEARS.append(int(match.group()))

# Sort the years to maintain chronological order
YEARS.sort()

if not YEARS:
    raise ValueError("No years found in jpg files")

In [7]:
print(YEARS)

[2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2022, 2023, 2024, 2025, 2026]


In [8]:
FESTIVAL_NAME = "electric_forest"

# 2. Collect DJ names from poster

In [9]:
import cv2
import pytesseract
import pandas as pd
import os
import re
import numpy as np

# Dictionary to store the lineup for each year
festival_lineups = {}

# Create extract folder if it doesn't exist
extract_folder = "../extract"
os.makedirs(extract_folder, exist_ok=True)

print("Starting OCR extraction...")

for year in YEARS:
    # Construct the filename based on the year
    filename = f"{year}_ef_lineup.jpg"
    file_path = os.path.join(data_folder, filename)
    
    if not os.path.exists(file_path):
        print(f"Warning: File not found for year {year}: {filename}")
        continue
        
    # Read the image
    img = cv2.imread(file_path)
    
    if img is None:
        print(f"Error reading image for year {year}")
        continue

    # --- Preprocessing ---
    # 1. Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 2. Resize (Upscale)
    # Tesseract works better on larger text. We double the image size.
    scale_percent = 200 
    width = int(img.shape[1] * scale_percent / 100)
    height = int(img.shape[0] * scale_percent / 100)
    dim = (width, height)
    resized = cv2.resize(gray, dim, interpolation = cv2.INTER_CUBIC)

    # 3. Adaptive Thresholding
    # This is crucial for posters with complex/colorful backgrounds.
    # It calculates the threshold for small regions of the image.
    thresh = cv2.adaptiveThreshold(resized, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                   cv2.THRESH_BINARY, 31, 15)
    
    # 4. Invert if necessary
    # Tesseract expects black text on white background.
    # If the image is mostly black (white text on dark background), we invert it.
    if cv2.countNonZero(thresh) < (thresh.size / 2):
        thresh = cv2.bitwise_not(thresh)

    # --- Extraction ---
    try:
        # --psm 6: Assume a single uniform block of text.
        text = pytesseract.image_to_string(thresh, config='--psm 6')
    except Exception as e:
        print(f"OCR failed for {year}: {e}")
        continue
    
    # --- Cleaning ---
    cleaned_names = []
    
    # Keywords to exclude (dates, locations, website info, common poster text)
    exclude_keywords = ["JUNE", "JULY", "AUGUST", "SEPTEMBER", "ROTHBURY", "MICHIGAN", 
                        "FESTIVAL", "COM", "TICKETS", "WWW", "PRESENTS", "FEATURING", 
                        "ANNUAL", "WEEKEND", "CAMPING", "ARTS"]
    
    for line in text.split('\n'):
        # Skip lines containing excluded keywords (case insensitive)
        if any(keyword in line.upper() for keyword in exclude_keywords):
            continue
            
        # Split on common separators found in posters:
        # • (bullet), | (pipe), :: (double colon), - (hyphen with spaces), . (dot with spaces)
        # We also handle cases where OCR reads a bullet as a period or other symbol
        parts = re.split(r"•|\||::|\s-\s|\s\.\s|\s\*\s", line)
        
        for part in parts:
            name = part.strip()
            # Filter out empty lines, very short strings, and pure numbers
            if len(name) > 2 and not name.isdigit():
                # Remove common OCR noise characters from the start/end
                name = name.strip(".,-_*~|[](){}")
                if len(name) > 2:
                    cleaned_names.append(name)
    
    festival_lineups[year] = cleaned_names
    
    # Save to CSV
    csv_filename = f"{year}_ef_artists.csv"
    csv_path = os.path.join(extract_folder, csv_filename)
    
    df = pd.DataFrame(cleaned_names, columns=["Artist"])
    df.to_csv(csv_path, index=False)
    
    print(f"Year {year}: Extracted {len(cleaned_names)} artists. Saved to {csv_path}")

print("\nExtraction complete.")
# Display a sample from the first available year
if YEARS:
    first_year = YEARS[0]
    print(f"\nSample extracted text for {first_year}:")
    print(festival_lineups[first_year][:10])

Starting OCR extraction...
Year 2011: Extracted 102 artists. Saved to ../extract/2011_ef_artists.csv
Year 2012: Extracted 50 artists. Saved to ../extract/2012_ef_artists.csv
Year 2013: Extracted 86 artists. Saved to ../extract/2013_ef_artists.csv
Year 2014: Extracted 88 artists. Saved to ../extract/2014_ef_artists.csv
Year 2015: Extracted 76 artists. Saved to ../extract/2015_ef_artists.csv
Year 2016: Extracted 56 artists. Saved to ../extract/2016_ef_artists.csv
Year 2017: Extracted 51 artists. Saved to ../extract/2017_ef_artists.csv
Year 2018: Extracted 69 artists. Saved to ../extract/2018_ef_artists.csv
Year 2019: Extracted 64 artists. Saved to ../extract/2019_ef_artists.csv
Year 2022: Extracted 64 artists. Saved to ../extract/2022_ef_artists.csv
Year 2023: Extracted 144 artists. Saved to ../extract/2023_ef_artists.csv
Year 2024: Extracted 52 artists. Saved to ../extract/2024_ef_artists.csv
Year 2025: Extracted 49 artists. Saved to ../extract/2025_ef_artists.csv
Year 2026: Extracted 3